In [ ]:
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
import os 
import sys
import pandas as pd
from sklearn.preprocessing import label_binarize
import pandas as pd
import numpy as np
import scipy.stats
from scipy import stats

In [ ]:
data_path = ''
train_csv = pd.read_csv(data_path+'')
test_csv = pd.read_csv(data_path+'')

print(train_csv.shape)
print(test_csv.shape)

In [ ]:
X_reg_train = train_csv[['reg_value']]
X_seg_train = train_csv[['seg_value']]
Y_train = train_csv['rank'].astype(int)+1
Y_train = label_binarize(Y_train, classes=[0, 1])

X_reg_test = test_csv[['reg_value']]
X_seg_test = test_csv[['seg_value']]
Y_test = test_csv['rank'].astype(int)+1
Y_test = label_binarize(Y_test, classes=[0, 1])


## model on seg and reg seperatly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold

base_classifier_lr = LogisticRegression(solver='lbfgs', max_iter=1000)


### construct the dataframe

In [ ]:
test_results = {
    'segmentation':{},
    'registration':{},
    'naive':{},
    'ensemble':{}
}

### Seg model

In [ ]:
X_seg_train_np = X_seg_train.values
model_seg = LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = np.zeros(5)  # Single AUC score per fold
acc_scores = np.zeros(5)
recall_scores = np.zeros(5)
specificity_scores = np.zeros(5)
precision_scores = np.zeros(5)
f1_scores = np.zeros(5)


# Perform 5-fold cross-validation on the training data
for fold, (train_index, val_index) in enumerate(kf.split(X_seg_train_np,Y_train)):
    model_seg = SVC(kernel='linear', probability=True,class_weight='balanced')
    X_train_fold, X_val_fold = X_seg_train_np[train_index], X_seg_train_np[val_index]
    y_train_fold, y_val_fold = Y_train[train_index], Y_train[val_index]

    # Fit the model on the training fold
    model_seg.fit(X_train_fold, y_train_fold)

    # Get predictions
    y_pred_fold = model_seg.predict(X_val_fold)
    y_prob_fold = model_seg.predict_proba(X_val_fold)

    # Calculate metrics
    acc_scores[fold] = accuracy_score(y_val_fold, y_pred_fold)
    precision_scores[fold] = precision_score(y_val_fold, y_pred_fold, average='macro')
    recall_scores[fold] = recall_score(y_val_fold, y_pred_fold, average='macro')
    specificity_scores[fold] = recall_score(y_val_fold==0, y_pred_fold==0, average='macro')
    f1_scores[fold] = f1_score(y_val_fold, y_pred_fold, average='macro')
    
    # Calculate AUC
    # If y_val_fold is one-hot, convert to 1D
    if y_val_fold.ndim > 1 and y_val_fold.shape[1] == 2:
        y_val_fold_1d = y_val_fold[:, 1]
    else:
        y_val_fold_1d = y_val_fold

    # If y_prob_fold is 2D, use the probability for the positive class
    if y_prob_fold.ndim > 1 and y_prob_fold.shape[1] == 2:
        y_prob_fold_1d = y_prob_fold[:, 1]
    else:
        y_prob_fold_1d = y_prob_fold

    auc_scores[fold] = roc_auc_score(y_val_fold_1d, y_prob_fold_1d)




In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, recall_score, precision_score, f1_score

def bootstrap_test_ci(y_true, y_pred, y_prob, metric_func, n_bootstrap=1000):
    scores = []
    n = len(y_true)

    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)

        # Skip resample if only one class present (important for AUC)
        if metric_func == roc_auc_score:
            if len(np.unique(y_true[idx])) < 2:
                continue
            score = metric_func(y_true[idx], y_prob[idx])
        else:
            score = metric_func(y_true[idx], y_pred[idx])

        scores.append(score)

    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)

    return lower, upper

    return lower, upper
def specificity_metric(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)


In [ ]:
# Train the final model on the entire training dataset
model_seg = SVC(kernel='linear', probability=True,class_weight='balanced')
model_seg.fit(X_seg_train_np, Y_train)

# Get predicted probabilities for the test data
prob_seg_test = model_seg.predict_proba(X_seg_test)
Y_pred_seg_test = model_seg.predict(X_seg_test)

y_test_seg = Y_test

# get the classification report
print(classification_report(Y_test, Y_pred_seg_test,digits=3))

# Calculate Accuracy
accuracy = accuracy_score(Y_test, Y_pred_seg_test)

# Calculate Specificity (True Negative Rate)
# For binary classification, specificity = TN / (TN + FP)
cm = confusion_matrix(Y_test, Y_pred_seg_test)
tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)
precision = precision_score(Y_test, Y_pred_seg_test)
f1 = f1_score(Y_test, Y_pred_seg_test)
auc_score = roc_auc_score(Y_test, prob_seg_test[:, 1])
p_seg = prob_seg_test[:, 1]

acc_ci  = bootstrap_test_ci(Y_test, Y_pred_seg_test, prob_seg_test[:,1], accuracy_score)
sens_ci = bootstrap_test_ci(Y_test, Y_pred_seg_test, prob_seg_test[:,1], recall_score)
prec_ci = bootstrap_test_ci(Y_test, Y_pred_seg_test, prob_seg_test[:,1], precision_score)
f1_ci   = bootstrap_test_ci(Y_test, Y_pred_seg_test, prob_seg_test[:,1], f1_score)
auc_ci  = bootstrap_test_ci(Y_test, Y_pred_seg_test, prob_seg_test[:,1], roc_auc_score)
spec_ci = bootstrap_test_ci(Y_test, Y_pred_seg_test, prob_seg_test[:,1], specificity_metric)



In [ ]:
classification_report_dict = classification_report(Y_test, Y_pred_seg_test,digits=3,output_dict=True)
test_results['segmentation']['Accuracy'] = accuracy
test_results['segmentation']['Recall'] = classification_report_dict['macro avg']['recall']
test_results['segmentation']['Specificity'] = specificity
test_results['segmentation']['Precision'] = classification_report_dict['macro avg']['precision']
test_results['segmentation']['F1-Score'] = classification_report_dict['macro avg']['f1-score']
test_results['segmentation']['AUC'] = auc_score

In [ ]:
y_score = prob_seg_test[:, 1]

# Compute ROC curve and ROC area
fpr, tpr, thresholds = roc_curve(Y_test, y_score)
roc_auc = auc(fpr, tpr)
n_bootstraps = 1000
rng = np.random.RandomState(42)
bootstrapped_scores = []

for i in range(n_bootstraps):
    # bootstrap by sampling with replacement on the prediction indices
    indices = rng.randint(0, len(y_score), len(y_score))
    if len(np.unique(Y_test[indices])) < 2:
        # We need at least one positive and one negative sample for ROC AUC
        continue
    score = roc_auc_score(Y_test[indices], y_score[indices])
    bootstrapped_scores.append(score)

# Compute 2.5th and 97.5th percentiles for 95% CI
sorted_scores = np.array(bootstrapped_scores)
sorted_scores.sort()
ci_lower = sorted_scores[int(0.025 * len(sorted_scores))]
ci_upper = sorted_scores[int(0.975 * len(sorted_scores))]



# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.3f}, 95% CI: {ci_lower:.3f}-{ci_upper:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

### Reg model

In [ ]:
X_reg_train_np = X_reg_train.values

model_reg = SVC(kernel='linear', probability=True,class_weight='balanced')

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = np.zeros(5)  # Single AUC score per fold
acc_scores = np.zeros(5)
recall_scores = np.zeros(5)
specificity_scores = np.zeros(5)
precision_scores = np.zeros(5)
f1_scores = np.zeros(5)


# Perform 5-fold cross-validation on the training data
for fold, (train_index, val_index) in enumerate(kf.split(X_reg_train_np,Y_train)):
    X_train_fold, X_val_fold = X_reg_train_np[train_index], X_reg_train_np[val_index]
    y_train_fold, y_val_fold = Y_train[train_index], Y_train[val_index]
    model_reg = SVC(kernel='linear', probability=True,class_weight='balanced')
    # Fit the model on the training fold
    model_reg.fit(X_train_fold, y_train_fold)

    # Get predictions
    y_pred_fold = model_reg.predict(X_val_fold)
    y_prob_fold = model_reg.predict_proba(X_val_fold)

    # Calculate metrics
    acc_scores[fold] = accuracy_score(y_val_fold, y_pred_fold)
    precision_scores[fold] = precision_score(y_val_fold, y_pred_fold, average='macro')
    specificity_scores[fold] = recall_score(y_val_fold==0, y_pred_fold==0, average='macro')
    recall_scores[fold] = recall_score(y_val_fold, y_pred_fold, average='macro')
    f1_scores[fold] = f1_score(y_val_fold, y_pred_fold, average='macro')
    
    # Calculate AUC
    # If y_val_fold is one-hot, convert to 1D
    if y_val_fold.ndim > 1 and y_val_fold.shape[1] == 2:
        y_val_fold_1d = y_val_fold[:, 1]
    else:
        y_val_fold_1d = y_val_fold

    # If y_prob_fold is 2D, use the probability for the positive class
    if y_prob_fold.ndim > 1 and y_prob_fold.shape[1] == 2:
        y_prob_fold_1d = y_prob_fold[:, 1]
    else:
        y_prob_fold_1d = y_prob_fold

    auc_scores[fold] = roc_auc_score(y_val_fold_1d, y_prob_fold_1d)


In [ ]:
# Train the final model on the entire training dataset
Y_train = Y_train.ravel()
Y_test = Y_test.ravel()
X_reg_train_np = X_reg_train.values
X_reg_test_np = X_reg_test.values

model_reg = SVC(kernel='linear', probability=True,class_weight='balanced')
model_reg.fit(X_reg_train_np, Y_train)

# Get predicted probabilities for the test data
prob_reg_test = model_reg.predict_proba(X_reg_test)
Y_pred_reg_test = model_reg.predict(X_reg_test)

y_test_reg = Y_test

# get the classification report
print(classification_report(y_test_reg, Y_pred_reg_test, digits=3))
# Calculate Accuracy
accuracy = accuracy_score(Y_test, Y_pred_reg_test)

# Calculate Specificity (True Negative Rate)
# For binary classification, specificity = TN / (TN + FP)
cm = confusion_matrix(Y_test, Y_pred_reg_test)
tn, fp, fn, tp = cm.ravel()

# Point estimates
accuracy = accuracy_score(Y_test, Y_pred_reg_test)
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)
precision = precision_score(Y_test, Y_pred_reg_test)
f1 = f1_score(Y_test, Y_pred_reg_test)
auc_score = roc_auc_score(Y_test, prob_reg_test[:, 1])
p_reg = prob_reg_test[:, 1]

# Bootstrap CI
acc_ci  = bootstrap_test_ci(Y_test, Y_pred_reg_test, prob_reg_test[:,1], accuracy_score)
sens_ci = bootstrap_test_ci(Y_test, Y_pred_reg_test, prob_reg_test[:,1], recall_score)
prec_ci = bootstrap_test_ci(Y_test, Y_pred_reg_test, prob_reg_test[:,1], precision_score)
f1_ci   = bootstrap_test_ci(Y_test, Y_pred_reg_test, prob_reg_test[:,1], f1_score)
auc_ci  = bootstrap_test_ci(Y_test, Y_pred_reg_test, prob_reg_test[:,1], roc_auc_score)
spec_ci = bootstrap_test_ci(Y_test, Y_pred_reg_test, prob_reg_test[:,1], specificity_metric)



In [ ]:
classification_report_dict = classification_report(Y_test, Y_pred_reg_test,digits=3,output_dict=True)
test_results['registration']['Accuracy'] = accuracy
test_results['registration']['Recall'] = classification_report_dict['macro avg']['recall']
test_results['registration']['Specificity'] = specificity
test_results['registration']['Precision'] = classification_report_dict['macro avg']['precision']
test_results['registration']['F1-Score'] = classification_report_dict['macro avg']['f1-score']
test_results['registration']['AUC'] = auc_score

In [ ]:
y_score = prob_reg_test[:, 1]

# Compute ROC curve and ROC area
fpr, tpr, thresholds = roc_curve(Y_test, y_score)
roc_auc = auc(fpr, tpr)
n_bootstraps = 1000
rng = np.random.RandomState(42)
bootstrapped_scores = []

for i in range(n_bootstraps):
    # bootstrap by sampling with replacement on the prediction indices
    indices = rng.randint(0, len(y_score), len(y_score))
    if len(np.unique(Y_test[indices])) < 2:
        # We need at least one positive and one negative sample for ROC AUC
        continue
    score = roc_auc_score(Y_test[indices], y_score[indices])
    bootstrapped_scores.append(score)

# Compute 2.5th and 97.5th percentiles for 95% CI
sorted_scores = np.array(bootstrapped_scores)
sorted_scores.sort()
ci_lower = sorted_scores[int(0.025 * len(sorted_scores))]
ci_upper = sorted_scores[int(0.975 * len(sorted_scores))]



### Direct combined model

In [ ]:

X_seg_train_np = X_seg_train.values
X_reg_train_np = X_reg_train.values

X_combined_train = np.hstack((X_reg_train_np, X_seg_train_np))
X_combined_test = np.hstack((X_reg_test, X_seg_test))


model_naive = LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = np.zeros(5)  # Single AUC score per fold
acc_scores = np.zeros(5)
precision_scores = np.zeros(5)
specificity_scores = np.zeros(5)
recall_scores = np.zeros(5)
f1_scores = np.zeros(5)

# Perform 5-fold cross-validation on the training data
for fold, (train_index, val_index) in enumerate(kf.split(X_combined_train,Y_train)):
    X_train_fold, X_val_fold = X_combined_train[train_index], X_combined_train[val_index]
    y_train_fold, y_val_fold = Y_train[train_index], Y_train[val_index]
    model_naive = SVC(kernel='linear', probability=True,class_weight='balanced')
    # Fit the model on the training fold
    model_naive.fit(X_train_fold, y_train_fold)

    # Get predictions
    y_pred_fold = model_naive.predict(X_val_fold)
    y_prob_fold = model_naive.predict_proba(X_val_fold)

    # Calculate metrics
    acc_scores[fold] = accuracy_score(y_val_fold, y_pred_fold)
    precision_scores[fold] = precision_score(y_val_fold, y_pred_fold, average='macro')
    recall_scores[fold] = recall_score(y_val_fold, y_pred_fold, average='macro')
    specificity_scores[fold] = recall_score(y_val_fold==0, y_pred_fold==0, average='macro')
    f1_scores[fold] = f1_score(y_val_fold, y_pred_fold, average='macro')
    
    # Calculate AUC
    # If y_val_fold is one-hot, convert to 1D
    if y_val_fold.ndim > 1 and y_val_fold.shape[1] == 2:
        y_val_fold_1d = y_val_fold[:, 1]
    else:
        y_val_fold_1d = y_val_fold

    # If y_prob_fold is 2D, use the probability for the positive class
    if y_prob_fold.ndim > 1 and y_prob_fold.shape[1] == 2:
        y_prob_fold_1d = y_prob_fold[:, 1]
    else:
        y_prob_fold_1d = y_prob_fold

    auc_scores[fold] = roc_auc_score(y_val_fold_1d, y_prob_fold_1d)



In [ ]:
# Train the final model on the entire training dataset


model_naive = SVC(kernel='linear', probability=True,class_weight='balanced')
model_naive.fit(X_combined_train, Y_train)

# Get predicted probabilities for the test data
prob_naive_test = model_naive.predict_proba(X_combined_test)
Y_pred_naive_test = model_naive.predict(X_combined_test)

# get the classification report
print(classification_report(Y_test, Y_pred_naive_test, digits=3))


# Calculate Specificity (True Negative Rate)
# For binary classification, specificity = TN / (TN + FP)
cm = confusion_matrix(Y_test, Y_pred_naive_test)
tn, fp, fn, tp = cm.ravel()

# Calculate Accuracy
accuracy = accuracy_score(Y_test, Y_pred_naive_test)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn)
precision = precision_score(Y_test, Y_pred_naive_test)
f1 = f1_score(Y_test, Y_pred_naive_test)
auc_score = roc_auc_score(Y_test, prob_naive_test[:, 1])
p_naive = prob_naive_test[:, 1]

acc_ci  = bootstrap_test_ci(Y_test, Y_pred_naive_test, prob_naive_test[:,1], accuracy_score)
auc_ci  = bootstrap_test_ci(Y_test, Y_pred_naive_test, prob_naive_test[:,1], roc_auc_score)
spec_ci = bootstrap_test_ci(Y_test, Y_pred_naive_test, prob_naive_test[:,1], specificity_metric)
sens_ci = bootstrap_test_ci(Y_test, Y_pred_naive_test, prob_naive_test[:,1], recall_score)
prec_ci = bootstrap_test_ci(Y_test, Y_pred_naive_test, prob_naive_test[:,1], precision_score)
f1_ci   = bootstrap_test_ci(Y_test, Y_pred_naive_test, prob_naive_test[:,1], f1_score)




In [ ]:

classification_report_dict = classification_report(Y_test, Y_pred_naive_test,digits=3,output_dict=True)
test_results['naive']['Accuracy'] = accuracy
test_results['naive']['Recall'] = classification_report_dict['macro avg']['recall']
test_results['naive']['Specificity'] = specificity
test_results['naive']['Precision'] = classification_report_dict['macro avg']['precision']
test_results['naive']['F1-Score'] = classification_report_dict['macro avg']['f1-score']
test_results['naive']['AUC'] = auc_score

In [ ]:
y_score = prob_naive_test[:, 1]

# Compute ROC curve and ROC area
fpr, tpr, thresholds = roc_curve(Y_test, y_score)
roc_auc = auc(fpr, tpr)
n_bootstraps = 10000
rng = np.random.RandomState(42)
bootstrapped_scores = []

for i in range(n_bootstraps):
    # bootstrap by sampling with replacement on the prediction indices
    indices = rng.randint(0, len(y_score), len(y_score))
    if len(np.unique(Y_test[indices])) < 2:
        # We need at least one positive and one negative sample for ROC AUC
        continue
    score = roc_auc_score(Y_test[indices], y_score[indices])
    bootstrapped_scores.append(score)

# Compute 2.5th and 97.5th percentiles for 95% CI
sorted_scores = np.array(bootstrapped_scores)
sorted_scores.sort()
ci_lower = sorted_scores[int(0.025 * len(sorted_scores))]
ci_upper = sorted_scores[int(0.975 * len(sorted_scores))]
# Plot ROC curve
# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.3f}, 95% CI: {ci_lower:.3f}-{ci_upper:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

### Ensemble model

In [ ]:
X_seg_train_np = X_seg_train.values
X_reg_train_np = X_reg_train.values

model_seg = LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')
model_reg =LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')
model_naive = LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')
model_ensemble =LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')
#model_ensemble = LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')

# Initialize KFold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Initialize arrays to store metrics
auc_scores = np.zeros(5)  # Single AUC score per fold
acc_scores = np.zeros(5)
precision_scores = np.zeros(5)
specificity_scores = np.zeros(5)
recall_scores = np.zeros(5)
f1_scores = np.zeros(5)

# Perform 5-fold cross-validation on the training data
for fold, (train_index, val_index) in enumerate(kf.split(X_seg_train_np,Y_train)):
    model_seg = SVC(kernel='linear', probability=True,class_weight='balanced')
    model_reg =SVC(kernel='linear', probability=True,class_weight='balanced')
    model_naive = SVC(kernel='linear', probability=True,class_weight='balanced')
    model_ensemble =LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')

    X_seg_train_fold, X_seg_val_fold = X_seg_train_np[train_index], X_seg_train_np[val_index]
    X_reg_train_fold, X_reg_val_fold = X_reg_train_np[train_index], X_reg_train_np[val_index]
    X_naive_train_fold, X_naive_val_fold = X_combined_train[train_index], X_combined_train[val_index]
    Y_train_fold, Y_val_fold = Y_train[train_index], Y_train[val_index]

    model_seg.fit(X_seg_train_fold, Y_train_fold)
    model_reg.fit(X_reg_train_fold, Y_train_fold)
    model_naive.fit(X_naive_train_fold, Y_train_fold)
    prob_seg_train = model_seg.predict_proba(X_seg_train_fold)
    prob_reg_train = model_reg.predict_proba(X_reg_train_fold)
    prob_naive_train = model_naive.predict_proba(X_naive_train_fold)
   
    # combine the probabilities
    combined_prob_train = np.hstack((prob_seg_train, prob_reg_train,prob_naive_train))
    model_ensemble.fit(combined_prob_train, Y_train_fold)

    # Get predicted probabilities for each class
    prob_seg_val = model_seg.predict_proba(X_seg_val_fold)
    prob_reg_val = model_reg.predict_proba(X_reg_val_fold)
    prob_naive_val = model_naive.predict_proba(X_naive_val_fold)

    # Combine probabilities
    combined_prob_val = np.hstack((prob_seg_val, prob_reg_val,prob_naive_val))
    y_prob_fold = model_ensemble.predict_proba(combined_prob_val)
    y_pred_fold = model_ensemble.predict(combined_prob_val)

    # Calculate metrics for multiclass
    acc_scores[fold] = accuracy_score(y_val_fold, y_pred_fold)
    precision_scores[fold] = precision_score(y_val_fold, y_pred_fold, average='macro')
    recall_scores[fold] = recall_score(y_val_fold, y_pred_fold, average='macro')
    f1_scores[fold] = f1_score(y_val_fold, y_pred_fold, average='macro')
    specificity_scores[fold] = recall_score(y_val_fold==0, y_pred_fold==0, average='macro')
    
    # Calculate AUC
    # If y_val_fold is one-hot, convert to 1D
    if y_val_fold.ndim > 1 and y_val_fold.shape[1] == 2:
        y_val_fold_1d = y_val_fold[:, 1]
    else:
        y_val_fold_1d = y_val_fold

    # If y_prob_fold is 2D, use the probability for the positive class
    if y_prob_fold.ndim > 1 and y_prob_fold.shape[1] == 2:
        y_prob_fold_1d = y_prob_fold[:, 1]
    else:
        y_prob_fold_1d = y_prob_fold

    auc_scores[fold] = roc_auc_score(y_val_fold_1d, y_prob_fold_1d)


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# ---- ensure 1D labels ----
Y_train = Y_train.ravel()
Y_test  = Y_test.ravel()

# ---- ensure numpy inputs (avoid feature-name warnings) ----
X_seg_train_np = np.asarray(X_seg_train_np)
X_reg_train_np = np.asarray(X_reg_train_np)
X_combined_train_np = np.asarray(X_combined_train)

X_seg_test_np = np.asarray(X_seg_test)
X_reg_test_np = np.asarray(X_reg_test)
X_combined_test_np = np.asarray(X_combined_test)

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def get_oof_and_test_proba(model_factory, X_train, y_train, X_test):
    """
    OOF probabilities for meta-train + averaged probabilities for test.
    """
    oof = np.zeros((len(y_train), 2))
    test_probs = []

    for tr_idx, va_idx in kf.split(X_train, y_train):
        m = model_factory()
        m.fit(X_train[tr_idx], y_train[tr_idx])
        oof[va_idx] = m.predict_proba(X_train[va_idx])
        test_probs.append(m.predict_proba(X_test))

    return oof, np.mean(test_probs, axis=0)

# base models (same settings as yours)
seg_factory   = lambda: SVC(kernel='linear', probability=True, class_weight='balanced')
reg_factory   = lambda: SVC(kernel='linear', probability=True, class_weight='balanced')
naive_factory = lambda: SVC(kernel='linear', probability=True, class_weight='balanced')

# OOF + test probabilities
oof_seg,  test_seg  = get_oof_and_test_proba(seg_factory,   X_seg_train_np, Y_train, X_seg_test_np)
oof_reg,  test_reg  = get_oof_and_test_proba(reg_factory,   X_reg_train_np, Y_train, X_reg_test_np)
oof_naiv, test_naiv = get_oof_and_test_proba(naive_factory, X_combined_train_np, Y_train, X_combined_test_np)

# stack features for meta model
stack_train = np.hstack([oof_seg, oof_reg, oof_naiv])
stack_test  = np.hstack([test_seg, test_reg, test_naiv])

# meta-learner
meta = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced')
meta.fit(stack_train, Y_train)

# test predictions
prob_ens_test = meta.predict_proba(stack_test)
pred_ens_test = meta.predict(stack_test)

In [ ]:
print(classification_report(Y_test, pred_ens_test, digits=3))

cm = confusion_matrix(Y_test, pred_ens_test)
tn, fp, fn, tp = cm.ravel()

accuracy    = accuracy_score(Y_test, pred_ens_test)
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)
precision   = precision_score(Y_test, pred_ens_test)
f1          = f1_score(Y_test, pred_ens_test)
auc_score   = roc_auc_score(Y_test, prob_ens_test[:, 1])
p_ens = prob_ens_test[:, 1]

# CIs
acc_ci  = bootstrap_test_ci(Y_test, pred_ens_test, prob_ens_test[:,1], accuracy_score)
sens_ci = bootstrap_test_ci(Y_test, pred_ens_test, prob_ens_test[:,1], recall_score)
spec_ci = bootstrap_test_ci(Y_test, pred_ens_test, prob_ens_test[:,1], specificity_metric)
prec_ci = bootstrap_test_ci(Y_test, pred_ens_test, prob_ens_test[:,1], precision_score)
f1_ci   = bootstrap_test_ci(Y_test, pred_ens_test, prob_ens_test[:,1], f1_score)
auc_ci  = bootstrap_test_ci(Y_test, pred_ens_test, prob_ens_test[:,1], roc_auc_score)



### Calibration and decision curve

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

meta_cal = CalibratedClassifierCV(meta, method='isotonic', cv=5)
meta_cal.fit(stack_train, Y_train)

p_ens_cal = meta_cal.predict_proba(stack_test)[:,1]

In [ ]:
from sklearn.metrics import brier_score_loss
y = Y_test.astype(int)  # 0/1
brier = {
    "Seg":      brier_score_loss(y, p_seg),
    "Reg":      brier_score_loss(y, p_reg),
    "Naive":    brier_score_loss(y, p_naive),
    "Ensemble": brier_score_loss(y, p_ens_cal),
}
print(brier)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve


def plot_calibration_curves_formatted(y, prob_dict, n_bins=4):

    plt.style.use("default")
    plt.rcParams.update({
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.facecolor": "white",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9
    })

   
    colors = {
        "Seg": "#3b4cc0",        
        "Reg": "#9fa8da",        
        "Naive": "#f7b5b5",      
        "Ensemble": "#c00000"    
    }

    plt.figure(figsize=(7,7))

    # Perfect calibration
    plt.plot([0.2, 1.0], [0.2, 1.0],
             linestyle=":",
             color="gray",
             linewidth=1.5,
             label="Perfect Calibration")

    for name, p in prob_dict.items():
        frac_pos, mean_pred = calibration_curve(
            y, p, n_bins=n_bins, strategy="quantile"
        )

        if name == "Ensemble":
            plt.plot(mean_pred, frac_pos,
                     marker="o",
                     markersize=6,
                     linewidth=2.8,
                     color=colors[name],
                     label=name)
        else:
            plt.plot(mean_pred, frac_pos,
                     marker="o",
                     markersize=6,
                     linestyle="--",
                     linewidth=1.8,
                     color=colors[name],
                     label=name)

    plt.xlim(0.2, 1.0)
    plt.ylim(0.2, 1.0)
    plt.gca().set_aspect('equal', adjustable='box')

    plt.xlabel("Predicted Probability")
    plt.ylabel("Observed Probability")
    plt.title("Calibration Curve (Test Set)")

    plt.legend(frameon=False)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


# Example usage
plot_calibration_curves_formatted(
    Y_test,
    {
        "Ensemble": p_ens_cal,
        "Seg": p_seg,
        "Reg": p_reg,
        "Naive": p_naive
    },
    n_bins=4
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def decision_curve(y_true, prob, thresholds):
    N = len(y_true)
    net_benefits = []

    for pt in thresholds:
        preds = (prob >= pt).astype(int)

        TP = np.sum((preds == 1) & (y_true == 1))
        FP = np.sum((preds == 1) & (y_true == 0))

        nb = (TP / N) - (FP / N) * (pt / (1 - pt))
        net_benefits.append(nb)

    return np.array(net_benefits)


# Threshold range (avoid 0 and 1)
thresholds = np.linspace(0.05, 0.95, 100)

# Ensemble net benefit
nb_ens = decision_curve(Y_test, p_ens_cal, thresholds)

# Treat-all strategy
prevalence = np.mean(Y_test)
nb_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))

# Treat-none strategy
nb_none = np.zeros_like(thresholds)

# Plot
plt.figure(figsize=(6,5))
plt.plot(thresholds, nb_ens, label="Ensemble", linewidth=2)
plt.plot(thresholds, nb_all, linestyle="--", label="Treat All")
plt.plot(thresholds, nb_none, linestyle=":", label="Treat None")

plt.xlabel("Threshold Probability")
plt.ylabel("Net Benefit")
plt.title("Decision Curve Analysis (Test Set)")
plt.legend(frameon=False)
plt.ylim(-0.1, 1)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def decision_curve(y_true, prob, thresholds):
    N = len(y_true)
    net_benefits = []

    for pt in thresholds:
        preds = (prob >= pt).astype(int)

        TP = np.sum((preds == 1) & (y_true == 1))
        FP = np.sum((preds == 1) & (y_true == 0))

        nb = (TP / N) - (FP / N) * (pt / (1 - pt))
        net_benefits.append(nb)

    return np.array(net_benefits)


thresholds = np.linspace(0.05, 0.95, 200)

# Compute curves
nb_seg   = decision_curve(Y_test, p_seg, thresholds)
nb_reg   = decision_curve(Y_test, p_reg, thresholds)
nb_naive = decision_curve(Y_test, p_naive, thresholds)
nb_ens   = decision_curve(Y_test, p_ens_cal, thresholds)

prevalence = np.mean(Y_test)
nb_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))
nb_none = np.zeros_like(thresholds)

# ---- Professional color palette ----
colors = {
    "Seg": "#3b4cc0",        
    "Reg": "#9fa8da",        
    "Naive": "#f7b5b5",      
    "Ensemble": "#c00000",    
    "TreatAll": "#999999",
    "TreatNone": "#000000"
}

# ---- Plot ----
plt.figure(figsize=(7,5))

plt.plot(thresholds, nb_ens,
         label="Ensemble",
         linewidth=2.8,
         color=colors["Ensemble"])

plt.plot(thresholds, nb_seg,
         linestyle="--",
         linewidth=1.8,
         color=colors["Seg"],
         label="Seg")

plt.plot(thresholds, nb_reg,
         linestyle="--",
         linewidth=1.8,
         color=colors["Reg"],
         label="Reg")

plt.plot(thresholds, nb_naive,
         linestyle="--",
         linewidth=1.8,
         color=colors["Naive"],
         label="Naive")

plt.plot(thresholds, nb_all,
         linestyle=":",
         linewidth=1.8,
         color=colors["TreatAll"],
         label="Treat All")

plt.plot(thresholds, nb_none,
         linestyle=":",
         linewidth=1.8,
         color=colors["TreatNone"],
         label="Treat None")

plt.xlabel("Threshold Probability")
plt.ylabel("Net Benefit")
plt.title("Decision Curve Analysis (Test Set)")
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Better axis limits
plt.xlim(0.1, 0.9)
plt.ylim(-0.05, 0.8)

plt.legend(frameon=False)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()